In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
  library(patchwork)
  library(pheatmap)
  library(grid)
  library(ggpubr)
})
root <- getwd()
oo <- readRDS(file.path(root, "Affirmseq_oocytes.rds"))
fd <- oo@misc$figure_data
cl <- c("1" = "#E64B35", "2" = "#4DAF4A", "3" = "#2DA8E0")
ts <- c("Other" = "grey80", "TS2" = "#2DA8E0", "TS4" = "#FF2B2B")
theme_set(theme_classic(base_size = 14))

In [ ]:
hm_cols <- colorRampPalette(c("#08306B", "#2DA8E0", "#F7F7F7", "#FF6B6B", "#B30000"))(100)
ann_cols <- list(pseudotime = colorRampPalette(c("grey85", "#2DA8E0"))(100))
pheatmap(fd$heatmap, color = hm_cols, cluster_rows = TRUE, cluster_cols = FALSE, show_rownames = FALSE, show_colnames = FALSE, annotation_col = fd$heatmap_annotation, annotation_colors = ann_cols, fontsize = 10, filename = file.path(root, "Affirmseq_oocytes_pseudotime_heatmap.pdf"), width = 7, height = 5.5)

s <- fd$schematic
p_s <- ggplot(s, aes(x, y)) + geom_segment(aes(x = 0.35, y = 0.15, xend = 1.18, yend = 0.68), arrow = arrow(length = unit(0.12, "in"))) + geom_segment(aes(x = 0.35, y = -0.15, xend = 1.18, yend = -0.68), arrow = arrow(length = unit(0.12, "in"))) + geom_point(size = 18, shape = 21, fill = "#E8A091", color = "#2DA8E0", stroke = 1.4) + geom_point(size = 7, shape = 21, fill = "#C5685F", color = NA) + geom_text(aes(label = label), vjust = c(2.5, -1.7, 2.5), size = 4) + coord_equal(xlim = c(-0.4, 2.2), ylim = c(-1.25, 1.25)) + theme_void()
ggsave(file.path(root, "Affirmseq_oocytes_schematic.pdf"), p_s, width = 3.2, height = 2.2, device = cairo_pdf)

u <- fd$umap
p_pt <- ggplot(u, aes(UMAP_1, UMAP_2)) + geom_segment(data = fd$trajectory_edges, aes(x = x, y = y, xend = xend, yend = yend), inherit.aes = FALSE, color = "black", linewidth = 0.6) + geom_point(aes(color = pseudotime), size = 2.4) + scale_color_gradientn(colors = c("grey80", "#2DA8E0"), name = "Pseudotime") + labs(x = "UMAP 1", y = "UMAP 2") + theme(axis.text = element_text(color = "black"))
ggsave(file.path(root, "Affirmseq_oocytes_pseudotime_umap.pdf"), p_pt, width = 4, height = 3, device = cairo_pdf)

p_ts <- ggplot(u, aes(UMAP_1, UMAP_2, color = terminal_focus)) + geom_point(size = 2.5) + scale_color_manual(values = ts) + labs(x = "UMAP 1", y = "UMAP 2", color = NULL) + theme(axis.text = element_text(color = "black"))
ggsave(file.path(root, "Affirmseq_oocytes_terminal_umap.pdf"), p_ts, width = 4.5, height = 3.5, device = cairo_pdf)

mb <- fd$terminal_marker_bars %>% mutate(gene = factor(gene, levels = rev(unique(gene))))
p_mb <- ggplot(mb, aes(avg_log2FC, gene, fill = terminal_state)) + geom_col(width = 0.75) + facet_wrap(~terminal_state, scales = "free_y") + scale_fill_manual(values = c("TS2" = "#2DA8E0", "TS4" = "#FF2B2B")) + labs(x = "Avg log2FC", y = "Gene") + guides(fill = "none") + theme(axis.text = element_text(color = "black"))
ggsave(file.path(root, "Affirmseq_oocytes_terminal_marker_bars.pdf"), p_mb, width = 4.6, height = 3.2, device = cairo_pdf)

In [ ]:
q <- fd$qc %>% pivot_longer(c(Gene, UMI), names_to = "metric", values_to = "value") %>% mutate(metric = factor(metric, c("Gene", "UMI"), c("nGenes per cell", "nUMIs per cell")))
p_qc <- ggplot(q, aes("Oocytes", value)) + geom_boxplot(width = 0.45, outlier.shape = NA, fill = "#DCEBF7", color = "#8CB6D8") + geom_jitter(width = 0.12, size = 0.8, color = "#2C77BF", alpha = 0.75) + facet_wrap(~metric, scales = "free_y", nrow = 1) + labs(x = NULL, y = NULL) + theme(strip.background = element_blank(), axis.text = element_text(color = "black"))
ggsave(file.path(root, "Affirmseq_oocytes_qc.pdf"), p_qc, width = 5.4, height = 3.8, device = cairo_pdf)

u <- fd$umap %>% mutate(cluster = factor(cluster, levels = c("1", "2", "3")))
p_cl <- ggplot(u, aes(UMAP_1, UMAP_2, color = cluster)) + geom_point(size = 2.5) + scale_color_manual(values = cl) + labs(x = "UMAP 1", y = "UMAP 2", color = NULL) + theme(axis.text = element_text(color = "black"))
ggsave(file.path(root, "Affirmseq_oocytes_cluster_umap.pdf"), p_cl, width = 4.2, height = 3.8, device = cairo_pdf)

su <- fd$score_umap %>% mutate(score_name = factor(score_name, c("GV_stage_score", "Growing_stage_score", "GVBD_stage_score"), c("GV stage score", "Growing stage score", "GVBD stage score")))
p_su <- ggplot(su, aes(UMAP_1, UMAP_2, color = score)) + geom_point(size = 2.2) + facet_wrap(~score_name, nrow = 1) + scale_color_gradientn(colors = c("grey85", "#2DA8E0"), name = "Score") + labs(x = "UMAP 1", y = "UMAP 2") + theme(axis.text = element_text(color = "black"), strip.background = element_blank())
ggsave(file.path(root, "Affirmseq_oocytes_stage_score_umap.pdf"), p_su, width = 8.5, height = 3.1, device = cairo_pdf)

sb <- fd$score_box %>% mutate(cluster = factor(as.integer(as.character(seurat_clusters)) + 1), score_name = factor(score_name, c("GV_stage_score", "Growing_stage_score", "GVBD_stage_score"), c("GV stage", "Growing stage", "GVBD stage")))
p_sb <- ggplot(sb, aes(cluster, score, fill = cluster)) + geom_boxplot(width = 0.65, outlier.shape = NA, alpha = 0.85) + geom_jitter(width = 0.12, size = 0.65, alpha = 0.35, color = "grey30") + facet_wrap(~score_name, nrow = 1, scales = "free_y") + scale_fill_manual(values = cl) + labs(x = NULL, y = "Module score") + guides(fill = "none") + theme(axis.text = element_text(color = "black"), strip.background = element_blank())
ggsave(file.path(root, "Affirmseq_oocytes_stage_score_boxplot.pdf"), p_sb, width = 7.2, height = 3.2, device = cairo_pdf)

gb <- fd$gene_box %>% mutate(terminal_state = factor(terminal_state, c("TS2", "TS4")))
st <- fd$gene_box_stats %>% mutate(group1 = "TS2", group2 = "TS4")
p_gb <- ggplot(gb, aes(terminal_state, expr, fill = terminal_state)) + geom_boxplot(width = 0.65, outlier.shape = NA, alpha = 0.85) + geom_jitter(aes(color = terminal_state), width = 0.14, size = 0.75, alpha = 0.45) + facet_wrap(~gene, scales = "free_y", nrow = 1) + ggpubr::stat_pvalue_manual(st, label = "p.adj.signif", xmin = "group1", xmax = "group2", y.position = "y.position", tip.length = 0.01, bracket.size = 0.25, size = 3, inherit.aes = FALSE) + scale_fill_manual(values = c("TS2" = "#2DA8E0", "TS4" = "#FF2B2B")) + scale_color_manual(values = c("TS2" = "#2DA8E0", "TS4" = "#FF2B2B")) + labs(x = NULL, y = "Expression") + guides(fill = "none", color = "none") + theme(axis.text = element_text(color = "black"), strip.background = element_blank())
ggsave(file.path(root, "Affirmseq_oocytes_TS2_TS4_gene_boxplot.pdf"), p_gb, width = 9.5, height = 3.2, device = cairo_pdf)